# 04 — Arazi Örtüsü Değişimi ve İklim İlişkisi

Bu notebook AS1–AS4 alt sorularının sayısal sonuçlarını üretir.

**Bölüm A — Haritalar (Earth Engine):**
1. Faz 3'te seçilen modelin ayarlarıyla Earth Engine'de bir sınıflandırıcı eğitilir (karar A). Bu model Colab'daki en iyi modelle **aynı test noktalarında** karşılaştırılır.
2. **Örneklem tabanlı sınıflandırma:** Her 1 km hücrede düzenli aralıklı 9 nokta (3 × 3, ~333 m arayla) seçilir. Earth Engine yalnızca bu noktalardaki 2014 ve 2024 bant/indeks değerlerini Drive'a yazar. Sınıflandırmayı Colab'da **Faz 3'ün en iyi modeli** yapar. İl il çalışılır, önce Artvin pilot olarak çalışır ve kota gösterilir.
3. Hücredeki sınıf oranları bu 9 noktadan, geçiş matrisi (2014'te X olan nokta 2024'te ne oldu?) ilin bütün noktalarından tahmin edilir.

**Bölüm B — Analiz (Colab, Earth Engine yok):**
4. Sınıf alanları ve değişim (%).
5. MODIS LST, ERA5-Land ve CHIRPS şehir serilerinde Mann-Kendall trend testi ve Sen eğimi.
6. 1 km hücrelerde Δ(arazi örtüsü) ile ΔLST ve ΔNDMI arasında Spearman korelasyonu.
7. LST regresyonu (Random Forest Regressor, mekânsal blok CV, SHAP).
8. Kentsel ısı adası (İstanbul, İzmir).
9. Alt soru özet tablosu.

> **Önemli yorum kuralı:** Bütün bulgular **ilişki (korelasyon)** düzeyindedir. "Yapılaşma sıcaklığı artırdı" değil, "yapılaşmanın arttığı hücrelerde sıcaklık artışı daha fazla" deriz.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Repo herkese açık olduğu için **token gerekmez**. Colab'ın **Gizli anahtarlar (Secrets)** bölümünde `GITHUB_TOKEN` tanımlıysa yine kullanılır (repo gizli yapılırsa gerekir); token ekrana yazdırılmaz.

In [ ]:
!pip install -q earthengine-api geemap pymannkendall shap

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
GITHUB_BRANCH = "main"   # kendi dalınızdaki kodu çalıştırmak için dal adını yazın (ör. "amine/geoclimate-uyum")
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        mesaj = r.stderr or r.stdout
        raise RuntimeError("git hatası: " + (mesaj.replace(token, "***") if token else mesaj))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Repo herkese açık: token gerekmez. Colab Secrets'ta GITHUB_TOKEN varsa kullanılır (repo gizli yapılırsa gerekir).
    try:
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        token = ""
    adres = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
    if not os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["clone", "-q", adres, CODE_DIR], token)
    # Seçilen dalın en güncel hâline geç (Colab kopyasındaki yerel değişiklikler atılır)
    _git(["-C", CODE_DIR, "fetch", "-q", "origin", GITHUB_BRANCH], token)
    _git(["-C", CODE_DIR, "checkout", "-q", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| dal:", GITHUB_BRANCH if IN_COLAB else "yerel", "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import ee
from src import gee_utils as gu
importlib.reload(gu)
gu.init_ee()

In [ ]:
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import ml_utils as mu
importlib.reload(mu)

FORCE_RECOMPUTE = False
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
T, F, M = config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR
RAW_GEE = f"{config.DATA_RAW}/gee"
os.makedirs(RAW_GEE, exist_ok=True)
C = config.COLLECTIONS
CITY_LIST = list(config.CITIES)
LABELS = sorted(config.CLASSES)
CLASS_NAMES = [config.CLASSES[k] for k in LABELS]
Y0, Y1 = config.CHANGE_YEARS
FEATS = config.MODEL_FEATURES

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

def set_props(props):
    return lambda f: f.set(props)

geoms = {c: gu.city_geometry(c) for c in CITY_LIST}
grid_proj = ee.Projection(config.GRID_CRS).atScale(config.GRID_SIZE_M)

samples = pd.read_csv(f"{config.DATA_PROCESSED}/samples_2021.csv")
split = pd.read_csv(f"{config.DATA_PROCESSED}/split_assignments.csv")[["sample_id", "split"]]
samples = samples.merge(split, on="sample_id")
meta = json.load(open(f"{M}/best_model.json", encoding="utf-8"))
print("Faz 3 en iyi model:", meta["best_model"])

## A1. Earth Engine sınıflandırıcısı
Earth Engine'in hazır sınıflandırıcıları scikit-learn ile birebir aynı değildir. Eşleme şöyle yapılıyor:
- En iyi model **Random Forest** ise `smileRandomForest` (aynı ağaç sayısı, yaprak büyüklüğü ve her bölünmede denenen özellik sayısı).
- En iyi model **XGBoost** ise `smileGradientTreeBoost` (aynı ağaç sayısı, öğrenme oranı, alt örnekleme; derinlik → en fazla yaprak sayısı).
- En iyi model **SVM veya MLP** ise Earth Engine'de birebir karşılığı olmadığı için doğrulama kümesinde en iyi olan **ağaç modeli** kullanılır ve bu açıkça yazılır.

Sınıflandırıcı, Faz 3'teki **eğitim** noktalarıyla eğitiliyor. Noktaların özellik değerleri zaten tabloda olduğu için Earth Engine'de görüntüden yeniden örnekleme yapılmıyor, bu yüzden kota harcanmıyor.

In [ ]:
val = pd.read_csv(f"{T}/model_karsilastirma_dogrulama.csv", index_col=0)
EE_MODEL = meta["best_model"] if meta["best_model"] in ("RandomForest", "XGBoost") \
    else val.loc[["RandomForest", "XGBoost"], "f1_macro"].idxmax()
ee_params = meta["all_params"][EE_MODEL]
print("Earth Engine'de kullanılan model:", EE_MODEL, ee_params,
      "" if EE_MODEL == meta["best_model"] else f"(Colab'daki en iyi model {meta['best_model']} GEE'de yok)")

train = samples[samples["split"] == "train"]
test = samples[samples["split"] == "test"]
train_fc = gu.df_to_fc(train, ["class"] + FEATS)
ee_clf = gu.make_ee_classifier(EE_MODEL, ee_params).train(
    features=train_fc, classProperty="class", inputProperties=FEATS)

## A2. Earth Engine modeli ile Colab modeli: aynı test noktalarında karşılaştırma
Haritaları Earth Engine modeli üreteceği için bu modelin Faz 3'teki modelle benzer başarıda olduğunu göstermemiz gerekir. Test noktaları Earth Engine'de sınıflandırılıyor; Accuracy, macro-F1 ve Kappa yan yana konuyor. Hesap tablodaki değerler üzerinden yapıldığı için hafif.

macro-F1 farkı %2 puandan büyükse notebook uyarı verir.

In [ ]:
cmp_csv = f"{T}/gee_vs_colab_karsilastirma.csv"
pred_csv = f"{config.DATA_PROCESSED}/test_predictions_gee.csv"
if os.path.exists(pred_csv) and not FORCE_RECOMPUTE:
    gee_pred = pd.read_csv(pred_csv)
else:
    test_fc = gu.df_to_fc(test, ["sample_id"] + FEATS)
    res = test_fc.classify(ee_clf, "pred")
    got = ee.Dictionary({"id": res.aggregate_array("sample_id"), "pred": res.aggregate_array("pred")}).getInfo()
    ids, out = got["id"], got["pred"]
    gee_pred = pd.DataFrame({"sample_id": ids, "pred_gee": out})
    gee_pred.to_csv(pred_csv, index=False)

colab = pd.read_csv(f"{config.DATA_PROCESSED}/test_predictions.csv")
both = colab.merge(gee_pred, on="sample_id")
rows = []
for label, col in [(f"Colab {meta['best_model']}", f"pred_{meta['best_model']}"),
                   (f"Colab {EE_MODEL}", f"pred_{EE_MODEL}"),
                   (f"Earth Engine {EE_MODEL}", "pred_gee")]:
    m = mu.compute_metrics(both["class"], both[col], None, LABELS)
    rows.append({"model": label, "accuracy": m["accuracy"], "f1_macro": m["f1_macro"], "kappa": m["kappa"]})
gee_cmp = pd.DataFrame(rows).drop_duplicates("model").set_index("model").round(4)
gee_cmp["n_test"] = len(both)
gee_cmp["harita_uyumu_colab_en_iyi"] = (both["pred_gee"] == both[f"pred_{meta['best_model']}"]).mean().round(4)
gee_cmp.to_csv(cmp_csv, encoding="utf-8-sig")
display(gee_cmp)

diff = gee_cmp.loc[f"Colab {meta['best_model']}", "f1_macro"] - gee_cmp.loc[f"Earth Engine {EE_MODEL}", "f1_macro"]
GEE_OK = abs(diff) <= config.GEE_VS_COLAB_WARN
print(f"macro-F1 farkı (Colab en iyi − GEE): {diff:+.4f} →",
      "kabul edilebilir" if GEE_OK else "BÜYÜK FARK: bu çıktıyı bize gönderin, haritalara geçmeden konuşalım.")

## A3. 2014 ve 2024: noktalarda bant değerleri (Earth Engine) → sınıflandırma (Colab)
**Neden değiştirdik?** İlk denemede tüm ili 30 m'de Earth Engine içinde sınıflandırdık. Bu çok pahalı oldu: Artvin'in yalnızca 2014 haritası 5,56 EECU-saat tuttu, 2024 haritası "Computation timed out", geçiş matrisi görevi "out of memory" hatası verdi. Asıl yük, yüzlerce ağaçlı modelin her 30 m pikselde ayrı ayrı çalıştırılmasıydı.

**Yeni yöntem (örneklem tabanlı alan tahmini):**
- Her 1 km hücreye **3 × 3 = 9 nokta** düşen düzenli bir nokta ağı kuruluyor (noktalar arası ~333 m).
- Earth Engine her noktada yalnızca **30 m kompozitin değerini** okuyor (11 model özelliği). Sınıflandırma yapmıyor, bu yüzden kota çok daha az.
- Noktalar Colab'da **Faz 3'ün en iyi modeliyle** (`best_model.joblib`) sınıflandırılıyor. Böylece haritalar raporda anlattığımız modelin kendisiyle üretiliyor; Earth Engine karşılığı gerekmiyor.
- **Hücredeki sınıf oranı** = hücredeki 9 noktadan kaçının o sınıf olduğu. **İl alanları** bu oranlardan hesaplanıyor.
- **Geçiş matrisi:** Aynı noktalar iki yılda da sınıflandırıldığı için 2014 → 2024 geçişi ilin bütün noktalarından (Artvin'de ~65.000) doğrudan sayılıyor.

**Bu yöntemin sınırı:** Bu bir **tahmin**dir, piksel piksel tam harita değildir. Bir hücrenin oranı 9 noktadan geldiği için 1/9'luk adımlarla (0, 0.11, 0.22 …) ilerler ve tek hücre düzeyinde gürültülüdür. İl toplamı ve geçiş oranları ise on binlerce noktadan geldiği için kararlıdır. Düzenli (sistematik) örnekleme alan tahmininde yaygın bir istatistiksel yöntemdir. Bu madde sınırlılıklarda yazılı.

**Model belirsizliği:** Model 2021 etiketiyle eğitildi. 2014 ve 2024'e uygulandığında doğruluğunu ölçemiyoruz, çünkü o yıllar için etiket yok. Yıllar arası atmosfer ve fenoloji farkları, gerçek değişim olmasa da sınıf değişimi gibi görünebilir. Bu bir sınırlılık maddesidir. Küçük değişimleri (birkaç puan) temkinli yorumlayın.

In [ ]:
GRID_TILE_SCALE = 16
PTS_SIDE = config.LC_POINTS_PER_CELL_SIDE                      # 3 → hücre başına 3 × 3 = 9 nokta
pt_proj = ee.Projection(config.GRID_CRS).atScale(config.GRID_SIZE_M / PTS_SIDE)
PT_COLS = ["city", "x", "y", "px", "py"] + FEATS

def classified(city, y):
    """Yalnızca görsel haritalar (aşağıda) için: Earth Engine karşılık modeliyle sınıflandırma."""
    return gu.classification_composite(geoms[city], y).classify(ee_clf).rename("lc")

def points_fc(city, y):
    """Nokta ağında 30 m kompozit değerleri. Sınıflandırma YOK (Colab'da yapılıyor).
    reproject(30 m): değer, noktanın düştüğü 30 m pikselden okunur (kaba piramit katmanından değil).
    x, y: noktanın düştüğü 1 km hücre (Faz 1 grid'iyle aynı kimlik); px, py: noktanın kendi kimliği."""
    feats30 = gu.classification_composite(geoms[city], y).reproject(crs=config.GRID_CRS,
                                                                     scale=config.LANDSAT_SCALE_M)
    cell_xy = ee.Image.pixelCoordinates(grid_proj).reproject(grid_proj)
    pt_xy = ee.Image.pixelCoordinates(pt_proj).rename(["px", "py"])
    img = feats30.addBands(cell_xy).addBands(pt_xy)
    return img.sample(region=geoms[city], projection=pt_proj, geometries=False,
                      dropNulls=False, tileScale=GRID_TILE_SCALE).map(set_props({"city": city}))

def city_exports(city):
    return {f"lc_points_{city}_{y}": (points_fc(city, y), PT_COLS) for y in config.CHANGE_YEARS}

### Pilot: Artvin
Kota kullanımı gösteriliyor. Kalan 6 il için kaba tahmin aylık kotanın %30'unu aşmazsa diğer iller otomatik başlar.

Eski yöntemden kalan `lc_grid_Artvin_2014.csv` dosyası artık kullanılmıyor (silmenize gerek yok).

In [ ]:
pilot = config.PILOT_CITY
KOTA_CSV = f"{T}/gee_kota_kullanimi.csv"
pilot_usage = gu.run_exports(city_exports(pilot), RAW_GEE, force=FORCE_RECOMPUTE)
if pilot_usage is not None:
    display(pilot_usage)

kota = pd.read_csv(KOTA_CSV)
names = set(city_exports(pilot))
pk = kota[kota["task"].isin(names) & (kota["state"] == "COMPLETED")].drop_duplicates("task", keep="last")
pilot_eecu = pd.to_numeric(pk["eecu_saat"], errors="coerce").sum()
km2 = {c: geoms[c].area(1000).divide(1e6).getInfo() for c in CITY_LIST}
est = pilot_eecu * sum(km2[c] for c in CITY_LIST if c != pilot) / km2[pilot]
used = pd.to_numeric(kota["eecu_saat"], errors="coerce").sum()
print(f"{pilot} (Faz 4): {pilot_eecu:.3f} EECU-saat | kalan 6 il KABA tahmin: {est:.2f} EECU-saat "
      f"(%{100 * est / config.GEE_MONTHLY_EECU_HOURS:.1f}) | şimdiye kadar toplam kayıtlı: {used:.2f}")
if names - set(pk["task"]) or pilot_eecu == 0:
    PILOT_OK = False
    print("UYARI: Pilot kota kullanımı kayıtta eksik. Tasks sayfasına bakın; makulse sonraki hücrede PILOT_OK = True yapın.")
else:
    PILOT_OK = (est <= config.PILOT_MAX_QUOTA_FRACTION * config.GEE_MONTHLY_EECU_HOURS
                and used + est <= 0.9 * config.GEE_MONTHLY_EECU_HOURS)
    print("→ Kalan iller otomatik başlatılacak." if PILOT_OK else
          "DURDU: Tahmin kota sınırını aşıyor. Bu çıktıyı bize gönderin.")

In [ ]:
STOP_AFTER_PILOT = False
if STOP_AFTER_PILOT or not PILOT_OK:
    print("Kalan iller başlatılmadı.")
else:
    rest = {}
    for city in CITY_LIST:
        if city != pilot:
            rest.update(city_exports(city))
    usage = gu.run_exports(rest, RAW_GEE, force=FORCE_RECOMPUTE)
    if usage is not None:
        display(usage)

## A4. Noktaları Colab'da sınıflandırma
- Her nokta 2014 ve 2024 için Faz 3'ün en iyi modeliyle sınıflandırılıyor (`c2014`, `c2024`).
- Kompozitte değeri olmayan noktalar (bulut vb.) sınıflandırılmıyor. `lc_valid_YYYY` = hücrede değeri olan noktaların oranı.
- Noktalar `data/processed/lc_noktalar_2014_2024.csv` dosyasına kaydediliyor; tekrar çalıştırınca yeniden hesaplanmıyor.
- Kontrol: noktaların hücre kimlikleri (`x, y`) Faz 1 grid'iyle eşleşmeli. Eşleşme %95'in altındaysa notebook durur.

In [ ]:
best_clf = joblib.load(f"{M}/best_model.joblib")
PTS_CSV = f"{config.DATA_PROCESSED}/lc_noktalar_2014_2024.csv"
grid = pd.read_csv(f"{config.DATA_PROCESSED}/grid_1km_2014_2024.csv")
grid[["x", "y"]] = grid[["x", "y"]].round(3)

def classify_points(city):
    d = None
    for y in config.CHANGE_YEARS:
        p = pd.read_csv(f"{RAW_GEE}/lc_points_{city}_{y}.csv")
        p[["x", "y", "px", "py"]] = p[["x", "y", "px", "py"]].round(3)
        ok = p[FEATS].notna().all(axis=1)
        p[f"c{y}"] = np.nan
        if ok.any():
            p.loc[ok, f"c{y}"] = best_clf.predict(p.loc[ok, FEATS])
        p = p[["city", "x", "y", "px", "py", f"c{y}"]]
        d = p if d is None else d.merge(p.drop(columns=["x", "y"]), on=["city", "px", "py"], how="inner")
    return d

if os.path.exists(PTS_CSV) and not FORCE_RECOMPUTE:
    pts = pd.read_csv(PTS_CSV)
else:
    pts = pd.concat([classify_points(c) for c in CITY_LIST], ignore_index=True)
    pts.to_csv(PTS_CSV, index=False)

# Hücre kimlikleri Faz 1 grid'iyle eşleşiyor mu?
keys = grid[["city", "x", "y"]].drop_duplicates()
match = pts[["city", "x", "y"]].drop_duplicates().merge(keys, how="left", indicator=True)
match_rate = (match["_merge"] == "both").mean()
nokta_ozet = pd.DataFrame({
    "nokta": pts.groupby("city").size(),
    "hucre": pts.groupby("city")[["x", "y"]].apply(lambda d: len(d.drop_duplicates())),
    f"siniflandirilan_{Y0}_yuzde": 100 * pts.groupby("city")[f"c{Y0}"].apply(lambda s: s.notna().mean()),
    f"siniflandirilan_{Y1}_yuzde": 100 * pts.groupby("city")[f"c{Y1}"].apply(lambda s: s.notna().mean()),
}).reindex(CITY_LIST).round(2)
nokta_ozet.to_csv(f"{T}/lc_nokta_ozet.csv", encoding="utf-8-sig")
display(nokta_ozet)
print(f"Faz 1 grid'iyle eşleşen hücre oranı: %{100 * match_rate:.1f}")
if match_rate < 0.95:
    raise RuntimeError("Hücre kimlikleri Faz 1 grid'iyle eşleşmiyor. Bu çıktıyı bize gönderin.")

### Değişim haritaları (görsel)
Rapor için her ilin 2014 ve 2024 sınıf haritası ile **değişim haritası** küçük boyutlu görüntü olarak indiriliyor. Değişim haritasında yalnızca sınıfı değişen pikseller boyanıyor (2024'teki yeni sınıfının rengiyle), değişmeyenler gri gösteriliyor.

2014 ve 2024 görselleri Earth Engine'deki **karşılık modelle** (A1) ve kaba çözünürlükte üretiliyor. Değişim haritası bu iki resimden Colab'da çıkarılıyor (Earth Engine'e ikinci kez gidilmiyor). Görseller **yalnızca görsel amaçlıdır**. Bütün sayısal sonuçlar A4'teki noktalardan (Faz 3'ün en iyi modeli) gelir. Bir ilin görseli üretilemezse (zaman aşımı) uyarı yazılır, notebook devam eder.

In [ ]:
import requests
from PIL import Image
from matplotlib.patches import Patch
map_dir = f"{F}/lc_maps"
os.makedirs(map_dir, exist_ok=True)
palette = [config.CLASS_COLORS[k].lstrip("#") for k in LABELS]
PNG_SIG = b"\x89PNG"

def is_png(path):
    if not os.path.exists(path):
        return False
    with open(path, "rb") as f:
        return f.read(4) == PNG_SIG

def thumb(img, city, path, vis):
    """Earth Engine'den küçük resim indirir. Sunucu resim yerine hata dönerse dosyaya YAZMAZ."""
    if is_png(path) and not FORCE_RECOMPUTE:
        return
    url = img.clip(geoms[city]).visualize(**vis).getThumbURL(
        {"region": geoms[city].bounds(), "dimensions": 1024, "format": "png"})
    r = requests.get(url, timeout=600)
    if r.status_code != 200 or not r.content.startswith(PNG_SIG):
        raise RuntimeError(f"Earth Engine resim döndürmedi: {r.text[:200]}")
    with open(path, "wb") as f:
        f.write(r.content)

def change_map(p0, p1, out):
    """Değişim haritası Colab'da, indirilen 2014 ve 2024 resimlerinden (Earth Engine'e gitmeden):
    rengi değişen piksel = 2024'teki yeni sınıfın rengi, değişmeyen = gri, il dışı = saydam."""
    a = np.asarray(Image.open(p0).convert("RGBA"))
    b = np.asarray(Image.open(p1).convert("RGBA"))
    inside = (a[..., 3] > 0) & (b[..., 3] > 0)
    changed = inside & (a[..., :3] != b[..., :3]).any(axis=-1)
    o = np.zeros_like(b)
    o[inside] = [224, 224, 224, 255]
    o[changed] = b[changed]
    Image.fromarray(o).save(out)

vis = {"min": 0, "max": len(LABELS) - 1, "palette": palette}
for city in CITY_LIST:
    p0, p1 = f"{map_dir}/{city}_{Y0}.png", f"{map_dir}/{city}_{Y1}.png"
    try:
        thumb(classified(city, Y0), city, p0, vis)
        thumb(classified(city, Y1), city, p1, vis)
        change_map(p0, p1, f"{map_dir}/{city}_degisim.png")
    except Exception as e:
        print(f"UYARI: {city} görseli üretilemedi ({str(e)[:200]}). Sayısal sonuçlar etkilenmez.")

fig, axes = plt.subplots(3, len(CITY_LIST), figsize=(3.2 * len(CITY_LIST), 10))
for j, city in enumerate(CITY_LIST):
    for i, (suffix, title) in enumerate([(Y0, str(Y0)), (Y1, str(Y1)), ("degisim", "değişim")]):
        p = f"{map_dir}/{city}_{suffix}.png"
        if is_png(p):
            axes[i, j].imshow(Image.open(p))
        axes[i, j].set_title(f"{city} {title}", fontsize=9); axes[i, j].axis("off")
fig.legend(handles=[Patch(color=config.CLASS_COLORS[k], label=config.CLASSES[k]) for k in LABELS]
           + [Patch(color="#e0e0e0", label="değişmedi")], loc="lower center", ncol=7, fontsize=9)
plt.subplots_adjust(bottom=0.06)
savefig("degisim_haritalari")

# Bölum B — Analiz
## B1. Sınıf alanları ve değişim (2014 → 2024)
Alan = Σ (hücredeki sınıf oranı × 1 km²). Değişim iki şekilde veriliyor:
- **Puan farkı (pp):** İlin yüzde kaçının o sınıf olduğu, 2014'ten 2024'e kaç puan değişti.
- **Göreli değişim (%):** Sınıfın kendi alanına göre değişim. Küçük sınıflarda (ör. su) göreli değişim büyük görünebilir, iki ölçüye birlikte bakın.

In [ ]:
# Hücre oranları noktalardan: lc{k}_YYYY = değeri olan noktalar içinde k sınıfının payı
frac = pts[["city", "x", "y"]].copy()
for y in config.CHANGE_YEARS:
    cy = pts[f"c{y}"]
    for k in LABELS:
        frac[f"lc{k}_{y}"] = (cy == k).astype(float).where(cy.notna())
    frac[f"lc_valid_{y}"] = cy.notna().astype(float)
lc = frac.groupby(["city", "x", "y"], as_index=False).mean()
cells = grid.merge(lc, on=["city", "x", "y"], how="left")
cells.to_csv(f"{config.DATA_PROCESSED}/grid_1km_lc_iklim.csv", index=False)
CELL_KM2 = (config.GRID_SIZE_M / 1000) ** 2

rows = []
for city, d in cells.groupby("city"):
    for k in LABELS:
        a0, a1 = d[f"lc{k}_{Y0}"].sum() * CELL_KM2, d[f"lc{k}_{Y1}"].sum() * CELL_KM2
        tot0 = d[[f"lc{j}_{Y0}" for j in LABELS]].sum().sum() * CELL_KM2
        tot1 = d[[f"lc{j}_{Y1}" for j in LABELS]].sum().sum() * CELL_KM2
        rows.append({"city": city, "sinif": config.CLASSES[k], f"alan_{Y0}_km2": a0, f"alan_{Y1}_km2": a1,
                     f"pay_{Y0}_yuzde": 100 * a0 / tot0, f"pay_{Y1}_yuzde": 100 * a1 / tot1,
                     "degisim_km2": a1 - a0, "degisim_pp": 100 * (a1 / tot1 - a0 / tot0),
                     "goreli_degisim_yuzde": 100 * (a1 - a0) / a0 if a0 > 0 else np.nan})
areas = pd.DataFrame(rows).round(3)
areas.to_csv(f"{T}/sinif_alanlari_degisim.csv", index=False, encoding="utf-8-sig")

pp = areas.pivot(index="city", columns="sinif", values="degisim_pp").reindex(index=CITY_LIST, columns=CLASS_NAMES)
plt.figure(figsize=(10, 5))
sns.heatmap(pp, annot=True, fmt="+.2f", cmap="RdBu_r", center=0, cbar_kws={"label": "puan (pp)"})
plt.title(f"Sınıf payındaki değişim, {Y0} → {Y1} (yüzde puan)"); plt.ylabel("")
savefig("sinif_degisim_pp")

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, y in zip(axes, config.CHANGE_YEARS):
    areas.pivot(index="city", columns="sinif", values=f"pay_{y}_yuzde").reindex(index=CITY_LIST, columns=CLASS_NAMES) \
        .plot(kind="bar", stacked=True, ax=ax, color=[config.CLASS_COLORS[k] for k in LABELS], width=0.8, legend=False)
    ax.set_title(f"{y}"); ax.set_ylabel("İl alanının yüzdesi (%)"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
savefig("sinif_alanlari_2014_2024")
areas[areas["city"].isin(CITY_LIST)].head(12)

## B2. Geçiş matrisleri
Satır = 2014 sınıfı, sütun = 2024 sınıfı. Hücredeki değer, 2014'te o sınıfta olan noktaların yüzde kaçının 2024'te sütundaki sınıfta olduğunu gösterir. Köşegen "değişmedi" demektir. Tahmini alan (km²) için nokta oranı il alanıyla çarpılıyor. Her ilde iki yılda da sınıflandırılmış bütün noktalar kullanılıyor (`n`).

In [ ]:
trans = pts.dropna(subset=[f"c{Y0}", f"c{Y1}"]).astype({f"c{Y0}": int, f"c{Y1}": int})
city_area = cells.groupby("city").size() * CELL_KM2
tm_rows = []
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, city in zip(axes.ravel(), CITY_LIST):
    d = trans[trans["city"] == city]
    ct = pd.crosstab(d[f"c{Y0}"], d[f"c{Y1}"]).reindex(index=LABELS, columns=LABELS, fill_value=0)
    row_pct = 100 * ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0)
    km2_est = ct / ct.values.sum() * city_area[city]
    for i in LABELS:
        for j in LABELS:
            tm_rows.append({"city": city, "sinif_2014": config.CLASSES[i], "sinif_2024": config.CLASSES[j],
                            "n_nokta": int(ct.loc[i, j]), "satir_yuzde": row_pct.loc[i, j],
                            "tahmini_km2": km2_est.loc[i, j]})
    sns.heatmap(row_pct, annot=True, fmt=".0f", cmap="Greens", vmin=0, vmax=100, cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 7})
    ax.set_title(f"{city} (n={len(d)})"); ax.set_xlabel(str(Y1)); ax.set_ylabel(str(Y0))
    ax.tick_params(axis="x", rotation=40, labelsize=7); ax.tick_params(axis="y", labelsize=7)
axes.ravel()[-1].axis("off")
savefig("gecis_matrisleri")
tm = pd.DataFrame(tm_rows).round(3)
tm.to_csv(f"{T}/gecis_matrisi.csv", index=False, encoding="utf-8-sig")

## B3. Şehir ölçeğinde iklim trendleri (2003–2024)
**Mann-Kendall testi:** Bir zaman serisinde artan ya da azalan **monoton bir eğilim** olup olmadığını sınar. Veri normal dağılmak zorunda değildir ve aykırı yıllara karşı dayanıklıdır; iklim serilerinde en yaygın trend testidir. p < 0.05 ise eğilim istatistiksel olarak anlamlı kabul edilir.

**Sen eğimi:** Trendin büyüklüğüdür (birim/yıl). Tüm yıl çiftleri arasındaki eğimlerin medyanıdır, bu yüzden tek bir sıcak yıldan etkilenmez.

ERA5-Land toprak nemi ~11 km çözünürlükte olduğu için **yalnızca burada**, şehir ölçeğinde kullanılıyor.

In [ ]:
import pymannkendall as mk
climate = pd.read_csv(f"{config.DATA_PROCESSED}/iklim_sehir_yillik.csv")
VARS = {"modis_lst_day": "MODIS yaz gündüz LST (°C)", "modis_lst_night": "MODIS yaz gece LST (°C)",
        "era5_t2m_summer": "ERA5 yaz 2 m sıcaklık (°C)", "era5_sm_summer": "ERA5 yaz toprak nemi (m³/m³)",
        "era5_sm_annual": "ERA5 yıllık toprak nemi (m³/m³)", "chirps_precip_annual": "CHIRPS yıllık yağış (mm)"}
rows = []
for city, d in climate.groupby("city"):
    d = d.sort_values("year")
    for v, label in VARS.items():
        s = d[v].dropna()
        r = mk.original_test(s.values)
        rows.append({"city": city, "degisken": v, "n_yil": len(s), "trend": r.trend, "p": r.p,
                     "sen_egimi_yil": r.slope, "sen_egimi_10yil": 10 * r.slope, "tau": r.Tau})
trend = pd.DataFrame(rows).round(5)
trend.to_csv(f"{T}/iklim_trend_mann_kendall.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, (v, label) in zip(axes.ravel(), VARS.items()):
    for city in CITY_LIST:
        d = climate[climate["city"] == city].sort_values("year")
        ax.plot(d["year"], d[v], marker="o", ms=3, lw=1.2, label=city)
    ax.set_title(label, fontsize=10); ax.set_xlabel("")
axes[0, 0].legend(fontsize=7, ncol=2)
savefig("iklim_serileri")

sig = trend.pivot(index="city", columns="degisken", values="sen_egimi_10yil").reindex(index=CITY_LIST, columns=list(VARS))
pv = trend.pivot(index="city", columns="degisken", values="p").reindex(index=CITY_LIST, columns=list(VARS))
stars = pd.DataFrame(np.where(pv.values < 0.05, "*", ""), index=sig.index, columns=sig.columns)
annot = sig.round(3).astype(str) + stars
plt.figure(figsize=(12, 4.5))
sns.heatmap(sig.div(sig.abs().max()), annot=annot, fmt="", cmap="RdBu_r", center=0, cbar=False)
plt.title("Sen eğimi (10 yılda değişim); * = Mann-Kendall p < 0.05  (renk: sütun içinde ölçekli)")
plt.ylabel(""); plt.xlabel("")
savefig("iklim_trend_ozet")
trend[trend["degisken"] == "modis_lst_day"]

## B4. 1 km hücrelerde değişim–iklim ilişkisi (Spearman)
Her hücre için 2024 − 2014 farkı (Δ) hesaplanıyor:
- **Arazi örtüsü:** Δorman, Δtarım, Δyapılaşma oranları (model haritalarından), Δsu (MNDWI su oranı; Faz 1'deki karar gereği iki yılda da aynı yöntem).
- **İndeks:** ΔNDVI.
- **İklim göstergesi:** ΔLST (Landsat, 30 m'den 1 km'ye ortalama), ΔMODIS LST, ΔNDMI (nem göstergesi).

**Spearman korelasyonu:** Pearson'dan farklı olarak değerlerin **sıralarını** kullanır. Doğrusal olmayan ama tek yönlü ilişkileri de yakalar ve aykırı değerlere dayanıklıdır.

**Yıl etkisi:** 2014 yazı ile 2024 yazı farklı hava koşullarına sahip olabilir. Bir ildeki tüm hücreleri aynı miktarda ısıtan bir fark Spearman korelasyonunu değiştirmez, çünkü korelasyon hücreleri **birbirine göre** karşılaştırır.

**p-değerleri hakkında:** Komşu hücreler birbirine benzer (mekânsal otokorelasyon). Hücreler bağımsız olmadığı için p-değerleri olduğundan küçük çıkar. Bu yüzden p'den çok **r'nin büyüklüğüne ve yönüne** bakın. Bu durum sınırlılıklarda yazılı.

Yalnızca iki yılda da sıcaklık verisinin en az %50'si olan ve sınıflandırılmış hücreler kullanılıyor.

In [ ]:
from scipy.stats import spearmanr
c = cells.copy()
ok = (c[f"st_valid_{Y0}"] >= 0.5) & (c[f"st_valid_{Y1}"] >= 0.5) & (c[f"lc_valid_{Y0}"] > 0.5) & (c[f"lc_valid_{Y1}"] > 0.5)
c = c[ok].copy()
c["d_orman"] = c[f"lc0_{Y1}"] - c[f"lc0_{Y0}"]
c["d_tarim"] = c[f"lc1_{Y1}"] - c[f"lc1_{Y0}"]
c["d_yapi"] = c[f"lc2_{Y1}"] - c[f"lc2_{Y0}"]
c["d_su"] = c[f"mndwi_water_{Y1}"] - c[f"mndwi_water_{Y0}"]
c["d_ndvi"] = c[f"NDVI_{Y1}"] - c[f"NDVI_{Y0}"]
c["d_lst"] = c[f"LST_{Y1}"] - c[f"LST_{Y0}"]
c["d_modis_lst"] = c[f"modis_lst_day_{Y1}"] - c[f"modis_lst_day_{Y0}"]
c["d_ndmi"] = c[f"NDMI_{Y1}"] - c[f"NDMI_{Y0}"]
c.to_csv(f"{config.DATA_PROCESSED}/grid_1km_delta.csv", index=False)

DRIVERS = {"d_orman": "Δorman", "d_tarim": "Δtarım", "d_yapi": "Δyapılaşma", "d_su": "Δsu", "d_ndvi": "ΔNDVI"}
TARGETS = {"d_lst": "ΔLST (Landsat)", "d_modis_lst": "ΔLST (MODIS)", "d_ndmi": "ΔNDMI (nem)"}
rows = []
for city, d in c.groupby("city"):
    for x, xl in DRIVERS.items():
        for y, yl in TARGETS.items():
            dd = d[[x, y]].dropna()
            if len(dd) < 30 or dd[x].std() == 0:
                continue
            r, p = spearmanr(dd[x], dd[y])
            rows.append({"city": city, "degisken": xl, "hedef": yl, "n_hucre": len(dd), "spearman_r": r, "p": p})
sp = pd.DataFrame(rows).round(4)
sp.to_csv(f"{T}/spearman_degisim_iklim.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
for ax, (y, yl) in zip(axes, TARGETS.items()):
    piv = sp[sp["hedef"] == yl].pivot(index="city", columns="degisken", values="spearman_r") \
            .reindex(index=CITY_LIST, columns=list(DRIVERS.values()))
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-0.6, vmax=0.6, ax=ax, cbar=ax is axes[-1])
    ax.set_title(yl); ax.set_xlabel(""); ax.set_ylabel("")
savefig("spearman_isi_haritasi")
print("Hücre sayıları:", c.groupby("city").size().to_dict())

In [ ]:
# Örnek dağılım grafikleri: alt sorulara en doğrudan bağlı ilişkiler
pairs = [("Artvin", "d_orman", "d_lst"), ("Sanliurfa", "d_ndvi", "d_lst"), ("Istanbul", "d_yapi", "d_lst"),
         ("Izmir", "d_yapi", "d_lst"), ("Konya", "d_su", "d_lst"), ("Konya", "d_tarim", "d_ndmi")]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (city, x, y) in zip(axes.ravel(), pairs):
    d = c[c["city"] == city][[x, y]].dropna()
    ax.hexbin(d[x], d[y], gridsize=40, cmap="viridis", mincnt=1, bins="log")
    r, p = spearmanr(d[x], d[y]) if len(d) > 2 else (np.nan, np.nan)
    ax.set_title(f"{city}: ρ = {r:.2f} (n={len(d)})", fontsize=10)
    ax.set_xlabel({**DRIVERS, **TARGETS}[x]); ax.set_ylabel({**DRIVERS, **TARGETS}[y])
    ax.axhline(0, color="w", lw=0.6); ax.axvline(0, color="w", lw=0.6)
savefig("degisim_iklim_dagilim")

## B5. LST regresyonu: hangi değişken yüzey sıcaklığıyla en çok ilişkili?
**Hedef:** 2024 yazı Landsat LST (her 1 km hücre). **Girdiler:** 2024 arazi örtüsü oranları (6 sınıf), NDVI, NDBI, MNDWI, NDMI ve rakım. Modeller: **Random Forest Regressor** ve **XGBoost Regressor**. Karşılaştırma için doğrusal (çoklu) regresyon da kuruluyor. SHAP ve özet tablolar RF üzerinden verilir; XGBoost, RF sonucunun modele özgü olup olmadığını kontrol eder.

**Mekânsal blok çapraz doğrulama:** Komşu hücreler birbirine çok benzer. Rastgele bölmede test hücrelerinin komşuları eğitimde kalır ve başarı yapay olarak yüksek çıkar (mekânsal veri sızıntısı). Bu yüzden harita 10 km × 10 km bloklara bölünüyor ve **aynı bloktaki hücreler hep aynı katmana** düşüyor (GroupKFold, 5 katman).

**Metrikler:**
- **R²:** LST'deki değişkenliğin yüzde kaçı açıklanıyor (1 = tamamı).
- **RMSE:** Tipik hata (°C). Büyük hataları daha çok cezalandırır.
- **MAE:** Ortalama mutlak hata (°C).

**SHAP:** Hangi girdinin tahmini ne yönde ve ne kadar etkilediğini gösterir.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_validate

REG_X = [f"lc{k}_{Y1}" for k in LABELS] + [f"{b}_{Y1}" for b in ["NDVI", "NDBI", "MNDWI", "NDMI"]] + ["elevation"]
REG_NAMES = {**{f"lc{k}_{Y1}": f"{config.CLASSES[k]} oranı" for k in LABELS},
             **{f"{b}_{Y1}": b for b in ["NDVI", "NDBI", "MNDWI", "NDMI"]}, "elevation": "Rakım"}
r = cells[(cells[f"st_valid_{Y1}"] >= 0.5) & (cells[f"lc_valid_{Y1}"] > 0.5)].dropna(subset=REG_X + [f"LST_{Y1}"])
Xr, yr = r[REG_X], r[f"LST_{Y1}"]
# x, y Earth Engine'in pixelCoordinates(grid_proj) çıktısı: metre DEĞİL, 1 km hücre birimi.
# Blok boyu bu birime çevriliyor (10 km = 10 hücre). Birim, komşu hücre aralığından okunuyor.
cell_step = np.median(np.diff(np.sort(r["x"].round(3).unique())))
block_units = cell_step * config.SPATIAL_BLOCK_M / config.GRID_SIZE_M
blocks = (r["city"] + "_" + (r["x"] // block_units).astype(int).astype(str) + "_"
          + (r["y"] // block_units).astype(int).astype(str))
print(f"Hücre aralığı: {cell_step:g} birim | blok: {block_units:g} birim | blok sayısı: {blocks.nunique()}")
if not 50 <= blocks.nunique() <= len(r) / 20:
    raise RuntimeError("Blok sayısı çok az: bloklar 10 km değil. Bu çıktıyı bize gönderin.")
gkf = GroupKFold(n_splits=5)
scoring = {"r2": "r2", "rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}
regs = {"RandomForestRegressor": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1,
                                                       random_state=config.RANDOM_STATE),
        "XGBRegressor": XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, tree_method="hist", n_jobs=-1,
                                     random_state=config.RANDOM_STATE),
        "LinearRegression": make_pipeline(StandardScaler(), LinearRegression())}
rows = []
for name, reg in regs.items():
    cvr = cross_validate(reg, Xr, yr, groups=blocks, cv=gkf, scoring=scoring, n_jobs=1)
    rows.append({"model": name, "R2_ort": cvr["test_r2"].mean(), "R2_std": cvr["test_r2"].std(),
                 "RMSE_ort_C": -cvr["test_rmse"].mean(), "MAE_ort_C": -cvr["test_mae"].mean(),
                 "n_hucre": len(r), "n_blok": blocks.nunique()})
reg_tab = pd.DataFrame(rows).round(4)
reg_tab.to_csv(f"{T}/lst_regresyon_metrikleri.csv", index=False, encoding="utf-8-sig")
reg_tab

In [ ]:
import shap
from sklearn.model_selection import cross_val_predict
rf_reg = regs["RandomForestRegressor"]
pred = cross_val_predict(rf_reg, Xr, yr, groups=blocks, cv=gkf, n_jobs=1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].hexbin(yr, pred, gridsize=60, cmap="viridis", mincnt=1, bins="log")
lims = [min(yr.min(), pred.min()), max(yr.max(), pred.max())]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set_xlabel("Gerçek LST (°C)"); axes[0].set_ylabel("Tahmin (°C, blok CV)")
axes[0].set_title(f"RF Regressor — R² = {reg_tab.loc[0, 'R2_ort']:.3f}, RMSE = {reg_tab.loc[0, 'RMSE_ort_C']:.2f} °C")

rf_reg.fit(Xr, yr)
joblib.dump(rf_reg, f"{M}/lst_rf_regressor.joblib", compress=3)
Xs = Xr.sample(min(2000, len(Xr)), random_state=config.RANDOM_STATE)
sv = shap.TreeExplainer(rf_reg).shap_values(Xs)
imp = pd.Series(np.abs(sv).mean(axis=0), index=[REG_NAMES[x] for x in REG_X]).sort_values()
direction = pd.Series([np.corrcoef(Xs[x], sv[:, i])[0, 1] for i, x in enumerate(REG_X)],
                      index=[REG_NAMES[x] for x in REG_X])
imp.plot(kind="barh", ax=axes[1], color=["#d73027" if direction[i] > 0 else "#4575b4" for i in imp.index])
axes[1].set_title("SHAP önemi (kırmızı: değer arttıkça LST artıyor, mavi: azalıyor)", fontsize=10)
axes[1].set_xlabel("ortalama |SHAP| (°C)")
savefig("lst_regresyon_shap")
shap_tab = pd.DataFrame({"ortalama_abs_shap_C": imp, "yon_korelasyonu": direction}).sort_values("ortalama_abs_shap_C", ascending=False).round(4)
shap_tab.to_csv(f"{T}/lst_regresyon_shap.csv", encoding="utf-8-sig")
shap_tab

## B6. Kentsel ısı adası (AS2: İstanbul, İzmir)
**Kentsel ısı adası (UHI):** Şehirlerin çevrelerindeki kırsal alanlardan daha sıcak olması. Bunun başlıca nedenleri beton ve asfaltın ısıyı depolaması, bitki örtüsünün az olması ve buharlaşmanın düşük olmasıdır.

**Tanım (1 km hücre):**
- **Kentsel:** yapılaşma oranı > %50
- **Kırsal:** yapılaşma oranı < %5, su oranı < %10 ve rakımı kentsel hücrelerin medyan rakımından en fazla 200 m farklı. Rakım koşulu, dağ serinliğinin ısı adasını olduğundan büyük göstermesini önler.

**UHI şiddeti** = kentsel hücrelerin ortalama LST'si − kırsal hücrelerin ortalama LST'si. Hem Landsat hem MODIS ile, iki yıl için ayrı ayrı hesaplanıyor.

In [ ]:
rows, box = [], []
for city in config.SUB_QUESTIONS["AS2"]:
    for y in config.CHANGE_YEARS:
        d = cells[(cells["city"] == city) & (cells[f"st_valid_{y}"] >= 0.5) & (cells[f"lc_valid_{y}"] > 0.5)]
        urban = d[d[f"lc2_{y}"] > config.UHI_URBAN_MIN]
        elev0 = urban["elevation"].median()
        rural = d[(d[f"lc2_{y}"] < config.UHI_RURAL_MAX) & (d[f"mndwi_water_{y}"] < config.UHI_WATER_MAX)
                  & ((d["elevation"] - elev0).abs() <= config.UHI_ELEV_TOL)]
        for src, col in [("Landsat", f"LST_{y}"), ("MODIS", f"modis_lst_day_{y}")]:
            rows.append({"city": city, "year": y, "kaynak": src, "n_kentsel": len(urban), "n_kirsal": len(rural),
                         "kentsel_LST": urban[col].mean(), "kirsal_LST": rural[col].mean(),
                         "UHI_C": urban[col].mean() - rural[col].mean()})
        box += [{"city": city, "year": y, "tip": "Kentsel", "LST": v} for v in urban[f"LST_{y}"]]
        box += [{"city": city, "year": y, "tip": "Kırsal", "LST": v} for v in rural[f"LST_{y}"]]
uhi = pd.DataFrame(rows).round(3)
uhi.to_csv(f"{T}/kentsel_isi_adasi.csv", index=False, encoding="utf-8-sig")

bx = pd.DataFrame(box)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, city in zip(axes, config.SUB_QUESTIONS["AS2"]):
    sns.boxplot(data=bx[bx["city"] == city], x="year", y="LST", hue="tip", ax=ax, fliersize=1,
                palette={"Kentsel": "#d73027", "Kırsal": "#1b7837"})
    u = uhi[(uhi["city"] == city) & (uhi["kaynak"] == "Landsat")].set_index("year")["UHI_C"]
    ax.set_title(f"{city} — UHI (Landsat): " + ", ".join(f"{y}: {v:+.1f} °C" for y, v in u.items()))
    ax.set_xlabel(""); ax.set_ylabel("Yaz LST (°C)")
savefig("kentsel_isi_adasi")
uhi

## B7. Su yüzeyi değişimi (AS3: Konya, Van, Samsun)
Açık su alanı (MNDWI > 0) 2014 → 2024 değişimi, modelin Su/Sulak sınıfı alanı ve aynı illerin yağış ve toprak nemi trendleri yan yana veriliyor. Faz 3'teki sulak alan bulgusu nedeniyle (otsu sulak alan çoğu zaman çayır/tarım olarak tahmin edilebiliyor) açık su için MNDWI'yi esas alın.

In [ ]:
water = pd.read_csv(f"{T}/su_alani_karsilastirma.csv").set_index("city")
cities3 = config.SUB_QUESTIONS["AS3"]
w = pd.DataFrame({
    f"mndwi_su_{Y0}_km2": water.loc[cities3, f"mndwi_{Y0}_km2"],
    f"mndwi_su_{Y1}_km2": water.loc[cities3, f"mndwi_{Y1}_km2"],
    "mndwi_degisim_yuzde": water.loc[cities3, "mndwi_degisim_yuzde_2014_2024"],
    "jrc_mndwi_iou_2021": water.loc[cities3, "iou_2021"],
    "model_su_sulak_degisim_km2": areas[(areas["sinif"] == config.CLASSES[5])].set_index("city").loc[cities3, "degisim_km2"],
})
for v in ["chirps_precip_annual", "era5_sm_summer"]:
    t_ = trend[trend["degisken"] == v].set_index("city")
    w[f"{v}_sen_10yil"] = t_.loc[cities3, "sen_egimi_10yil"]
    w[f"{v}_p"] = t_.loc[cities3, "p"]
w = w.round(4)
w.to_csv(f"{T}/su_degisim_ozet.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 4.5))
w[[f"mndwi_su_{Y0}_km2", f"mndwi_su_{Y1}_km2"]].plot(kind="bar", ax=ax, color=["#9ecae1", "#08519c"], width=0.7)
for i, v in enumerate(w["mndwi_degisim_yuzde"]):
    ax.text(i, w.iloc[i, :2].max(), f"{v:+.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Açık su alanı (km², MNDWI > 0)"); ax.set_title(f"Su yüzeyi {Y0} → {Y1}"); ax.tick_params(axis="x", rotation=0)
savefig("su_alani_degisim")
w

## B8. Alt soru özet tablosu
Her alt soru için ilgili sayısal sonuçlar ve grafik dosyaları otomatik dolduruluyor. **Bulgu** sütunu yalnızca ölçülen sayıyı tarif eder; yorumu siz yazacaksınız.

In [ ]:
def p_txt(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3g}"

def sp_val(city, drv, tgt):
    q = sp[(sp["city"] == city) & (sp["degisken"] == drv) & (sp["hedef"] == tgt)]
    return (f"ρ = {q['spearman_r'].iloc[0]:+.2f} ({p_txt(q['p'].iloc[0])}, n = {q['n_hucre'].iloc[0]})"
            if len(q) else "hesaplanamadı")

def pp_val(city, cls):
    q = areas[(areas["city"] == city) & (areas["sinif"] == config.CLASSES[cls])]
    return f"{q['degisim_pp'].iloc[0]:+.2f} pp ({q['degisim_km2'].iloc[0]:+.1f} km²)"

def tr_val(city, v):
    q = trend[(trend["city"] == city) & (trend["degisken"] == v)].iloc[0]
    return f"{q['sen_egimi_10yil']:+.3f}/10 yıl (MK p = {q['p']:.3g})"

L, MO, N = "ΔLST (Landsat)", "ΔLST (MODIS)", "ΔNDMI (nem)"
rows = []
for city in config.SUB_QUESTIONS["AS1"]:
    rows += [("AS1", city, f"Orman payı değişimi {Y0}→{Y1}", pp_val(city, 0), "sinif_degisim_pp.png"),
             ("AS1", city, "Δorman – ΔLST", sp_val(city, "Δorman", L), "spearman_isi_haritasi.png"),
             ("AS1", city, "ΔNDVI – ΔLST", sp_val(city, "ΔNDVI", L), "degisim_iklim_dagilim.png"),
             ("AS1", city, "MODIS yaz gündüz LST trendi", tr_val(city, "modis_lst_day"), "iklim_serileri.png")]
for city in config.SUB_QUESTIONS["AS2"]:
    u = uhi[(uhi["city"] == city) & (uhi["kaynak"] == "Landsat")].set_index("year")["UHI_C"]
    rows += [("AS2", city, f"Yapılaşma payı değişimi {Y0}→{Y1}", pp_val(city, 2), "sinif_degisim_pp.png"),
             ("AS2", city, "Δyapılaşma – ΔLST", sp_val(city, "Δyapılaşma", L), "spearman_isi_haritasi.png"),
             ("AS2", city, "UHI şiddeti (Landsat)", f"{Y0}: {u.get(Y0, np.nan):+.2f} °C, {Y1}: {u.get(Y1, np.nan):+.2f} °C",
              "kentsel_isi_adasi.png")]
for city in config.SUB_QUESTIONS["AS3"]:
    rows += [("AS3", city, f"Açık su alanı değişimi {Y0}→{Y1} (MNDWI)", f"{w.loc[city, 'mndwi_degisim_yuzde']:+.1f}%",
              "su_alani_degisim.png"),
             ("AS3", city, "Δsu – ΔLST", sp_val(city, "Δsu", L), "spearman_isi_haritasi.png"),
             ("AS3", city, "Δsu – ΔNDMI", sp_val(city, "Δsu", N), "spearman_isi_haritasi.png"),
             ("AS3", city, "ERA5 yaz toprak nemi trendi", tr_val(city, "era5_sm_summer"), "iklim_serileri.png"),
             ("AS3", city, "CHIRPS yıllık yağış trendi", tr_val(city, "chirps_precip_annual"), "iklim_serileri.png")]
for city in config.SUB_QUESTIONS["AS4"]:
    rows += [("AS4", city, f"Tarım payı değişimi {Y0}→{Y1}", pp_val(city, 1), "sinif_degisim_pp.png"),
             ("AS4", city, "Δtarım – ΔNDMI", sp_val(city, "Δtarım", N), "spearman_isi_haritasi.png"),
             ("AS4", city, "Δtarım – ΔLST", sp_val(city, "Δtarım", L), "spearman_isi_haritasi.png"),
             ("AS4", city, "ERA5 yaz toprak nemi trendi", tr_val(city, "era5_sm_summer"), "iklim_serileri.png")]
top = shap_tab.index[0]
rows.append(("AS1–AS4", "Tümü", "LST regresyonu (RF, mekânsal blok CV)",
             f"R² = {reg_tab.loc[0, 'R2_ort']:.3f}, RMSE = {reg_tab.loc[0, 'RMSE_ort_C']:.2f} °C, "
             f"MAE = {reg_tab.loc[0, 'MAE_ort_C']:.2f} °C; en etkili: {top}", "lst_regresyon_shap.png"))
summary = pd.DataFrame(rows, columns=["Alt soru", "Şehir", "Bulgu", "Sayısal değer", "Grafik dosyası"])
summary.to_csv(f"{T}/alt_soru_ozet_AS1_AS4.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
summary

## B9. Hipotez kanıt tablosu (H1–H5)
GeoClimate AI hipotezlerini mevcut analizlerle eşleştiriyoruz. Her hipotez için **7 ilin tamamında** ilgili Spearman korelasyonu (B4) alınıyor ve beklenen yönle karşılaştırılıyor.

| Hipotez | Kanıt | Beklenen |
|---|---|---|
| H1 Yapılaşma ↑ ↔ LST ↑ | Δyapılaşma – ΔLST; ayrıca kentsel ısı adası (B6) | ρ > 0 |
| H2 Orman/bitki ↓ ↔ LST ↑ | Δorman – ΔLST, ΔNDVI – ΔLST | ρ < 0 |
| H3 Tarım ↔ bitki/nem | Δtarım – ΔNDMI, Δtarım – ΔLST | ilişki var (yön belirtilmemiş) |
| H4 Su ↔ sıcaklık/nem | Δsu – ΔLST, Δsu – ΔNDMI | ilişki var (yön belirtilmemiş) |
| H5 İlişkiler bölgeye göre farklı | Aynı ilişkinin iller arasındaki ρ aralığı ve işaret değişimi | aralık geniş / işaret değişiyor |

**Güç sınıfları (yaygın kural):** |ρ| < 0.1 ihmal edilebilir, 0.1–0.3 zayıf, 0.3–0.5 orta, ≥ 0.5 güçlü. Hücre sayısı çok büyük olduğu için p-değerleri neredeyse hep < 0.001; otokorelasyon nedeniyle p'ye değil ρ'nun büyüklüğüne ve yönüne bakın.

**Karar sütunu bilerek boş:** "Desteklendi / kısmen desteklendi / desteklenmedi" yorumunu ekip yazmalı. Tablo yalnızca sayıları toplar. İlişki neden-sonuç değildir.

In [ ]:
def strength(r):
    a = abs(r)
    return "ihmal edilebilir" if a < 0.1 else "zayıf" if a < 0.3 else "orta" if a < 0.5 else "güçlü"

EVIDENCE = [  # (hipotez, sürücü, hedef, beklenen işaret: +1 / -1 / None = yön belirtilmemiş)
    ("H1", "Δyapılaşma", L, +1),
    ("H2", "Δorman", L, -1),
    ("H2", "ΔNDVI", L, -1),
    ("H3", "Δtarım", N, None),
    ("H3", "Δtarım", L, None),
    ("H4", "Δsu", L, None),
    ("H4", "Δsu", N, None),
]
rows = []
for h, drv, tgt, sign in EVIDENCE:
    for city in CITY_LIST:
        q = sp[(sp["city"] == city) & (sp["degisken"] == drv) & (sp["hedef"] == tgt)]
        if not len(q):
            continue
        r_, p_ = q["spearman_r"].iloc[0], q["p"].iloc[0]
        rows.append({"hipotez": h, "iliski": f"{drv} – {tgt}", "city": city, "rho": r_, "p": p_,
                     "n_hucre": q["n_hucre"].iloc[0], "guc": strength(r_),
                     "beklenen_yon": {1: "+", -1: "−", None: "belirtilmemiş"}[sign],
                     "yon_uyumlu": (np.sign(r_) == sign) if sign else np.nan,
                     "ana_odak_ili": city in config.SUB_QUESTIONS[config.HYPOTHESES[h][1]]})
kanit = pd.DataFrame(rows).round(4)
kanit.to_csv(f"{T}/hipotez_kanit_H1_H5.csv", index=False, encoding="utf-8-sig")

ozet_rows = []
for (h, rel), g in kanit.groupby(["hipotez", "iliski"], sort=False):
    anlamli = g[g["rho"].abs() >= 0.1]
    ozet_rows.append({
        "hipotez": h, "ifade": config.HYPOTHESES[h][0], "kanit": rel, "beklenen_yon": g["beklenen_yon"].iloc[0],
        "n_il": len(g), "il_rho_en_az_0.1": len(anlamli),
        "beklenen_yonde_il (|rho|>=0.1)": int((anlamli["yon_uyumlu"] == True).sum()) if g["beklenen_yon"].iloc[0] != "belirtilmemiş" else "—",
        "rho_min": g["rho"].min(), "rho_max": g["rho"].max(),
        "en_guclu_il": g.loc[g["rho"].abs().idxmax(), "city"],
        "karar_[EKİP_YAZACAK]": ""})
    # H5: aynı ilişki illere göre ne kadar değişiyor?
    ozet_rows.append({
        "hipotez": "H5", "ifade": config.HYPOTHESES["H5"][0], "kanit": f"{rel} (iller arası)", "beklenen_yon": "—",
        "n_il": len(g), "il_rho_en_az_0.1": len(anlamli), "beklenen_yonde_il (|rho|>=0.1)": "—",
        "rho_min": g["rho"].min(), "rho_max": g["rho"].max(),
        "en_guclu_il": f"aralık {g['rho'].max() - g['rho'].min():.2f}; işaret {'DEĞİŞİYOR' if (g['rho'] > 0.1).any() and (g['rho'] < -0.1).any() else 'aynı'}",
        "karar_[EKİP_YAZACAK]": ""})
u = uhi[uhi["kaynak"] == "Landsat"]
ozet_rows.append({"hipotez": "H1", "ifade": config.HYPOTHESES["H1"][0], "kanit": "Kentsel ısı adası (Landsat, B6)",
                  "beklenen_yon": "+", "n_il": u["city"].nunique(), "il_rho_en_az_0.1": "—",
                  "beklenen_yonde_il (|rho|>=0.1)": int((u.groupby("city")["UHI_C"].mean() > 0).sum()),
                  "rho_min": np.nan, "rho_max": np.nan,
                  "en_guclu_il": "; ".join(f"{c} {y}: {v:+.2f} °C" for c, y, v in u[["city", "year", "UHI_C"]].itertuples(index=False)),
                  "karar_[EKİP_YAZACAK]": ""})
hyp = pd.DataFrame(ozet_rows)
hyp["_s"] = hyp["hipotez"].str[1].astype(int)
hyp = hyp.sort_values("_s", kind="stable").drop(columns="_s").round(3)
hyp.to_csv(f"{T}/hipotez_ozet_H1_H5.csv", index=False, encoding="utf-8-sig")

piv = kanit.pivot_table(index="iliski", columns="city", values="rho").reindex(columns=CITY_LIST)
piv.index = [f"{kanit.loc[kanit['iliski'] == i, 'hipotez'].iloc[0]}: {i}" for i in piv.index]
plt.figure(figsize=(11, 4.5))
sns.heatmap(piv.sort_index(), annot=True, fmt="+.2f", cmap="RdBu_r", vmin=-0.6, vmax=0.6, cbar_kws={"label": "Spearman ρ"})
plt.title("H1–H5 kanıtı: değişim–iklim ilişkisi, 7 il"); plt.xlabel(""); plt.ylabel("")
savefig("hipotez_kanit_H1_H5")
hyp

## Faz 4 bitti mi?
- [ ] `gee_vs_colab_karsilastirma.csv`: Earth Engine modeli ile Colab modeli aynı test noktalarında
- [ ] `lc_nokta_ozet.csv` (eşleşme %95+), `sinif_alanlari_degisim.csv`, `gecis_matrisi.csv`, `degisim_haritalari.png`
- [ ] `iklim_trend_mann_kendall.csv`, `spearman_degisim_iklim.csv`, `lst_regresyon_metrikleri.csv`, `kentsel_isi_adasi.csv`
- [ ] `alt_soru_ozet_AS1_AS4.csv`: her alt soru için en az bir sayı ve grafik
- [ ] `hipotez_kanit_H1_H5.csv`, `hipotez_ozet_H1_H5.csv`: H1–H5 kanıt tablosu (karar sütununu ekip doldurur)

## Bu fazda ne yaptık, neden yaptık
- 2021 etiketiyle eğitilen modeli 2014 ve 2024'e uyguladık. Tüm ili Earth Engine'de sınıflandırmak kotayı aştığı için örneklem tabanlı yönteme geçtik: her 1 km hücrede 9 noktanın değerini Earth Engine'den aldık, Faz 3'ün en iyi modeliyle Colab'da sınıflandırdık. Sınıf alanlarını, değişimi ve geçiş matrisini bu noktalardan tahmin ettik.
- İklim göstergelerinin 22 yıllık eğilimini Mann-Kendall ve Sen eğimi ile ölçtük. Bu yöntemler normal dağılım varsaymaz ve aykırı yıllara dayanıklıdır.
- Arazi örtüsü değişimi ile sıcaklık ve nem değişimi arasındaki ilişkiyi 1 km hücrelerde Spearman korelasyonuyla ölçtük. Sonuçları neden-sonuç olarak değil, ilişki olarak raporluyoruz.
- LST'yi arazi örtüsü ve indekslerle tahmin eden bir regresyon modeli kurduk. Mekânsal blok CV ile komşu hücre sızıntısını önledik, SHAP ile hangi değişkenin etkili olduğunu gösterdik.
- İstanbul ve İzmir'de kentsel ısı adası şiddetini rakım kontrollü bir kırsal referansla iki yıl için hesapladık.